# 00 — Data Acquisition — Materials

Downloads the Australian Materials equity universe via yfinance (Yahoo Finance) using the provided source data "ASX_tickers". (This will need to be put in the source data folder).

In [2]:
## If not run before:
## !pip install yfinance


In [1]:
from pathlib import Path
import re, time, warnings, numpy as np, pandas as pd
import yfinance as yf
warnings.filterwarnings("ignore")
cwd=Path.cwd().resolve(); ROOT=cwd.parent if cwd.name.lower()=="notebooks" else cwd
RAW=ROOT/"data/raw"; INTERIM=ROOT/"data/interim"; SOURCE=ROOT/"source_data"; DIAG=ROOT/"results/diagnostics"
for p in [RAW,INTERIM,SOURCE,DIAG]: p.mkdir(parents=True,exist_ok=True)
START="2012-01-01"; END="2027-01-01"

## 1. Locate the ASX listed-companies source file

In [2]:
files=list(SOURCE.glob("*.csv"))+list(ROOT.glob("*.csv"))
if not files:
    raise FileNotFoundError("Place the ASX listed-companies CSV used in Financial Services into source_data/.")
src=files[0]
companies=pd.read_csv(src)
ticker_col=next((c for c in companies.columns if "code" in c.lower() or "ticker" in c.lower()),None)
if ticker_col is None: raise KeyError("Couldn't find ASX ticker/code column.")
companies["Ticker"]=companies[ticker_col].astype(str).str.strip().str.upper()
companies["Yahoo_Ticker"]=companies["Ticker"].where(companies["Ticker"].str.endswith(".AX"),companies["Ticker"]+".AX")
print(src,companies.shape)

C:\Users\Brian\Documents\Machine_Learning_Assignment\05_ASX_Materials_Volatility\source_data\asx_tickers.csv (1830, 6)


## 2. Retrieve Yahoo metadata and identify Materials securities

In [3]:
meta=[]
for i,t in enumerate(companies["Yahoo_Ticker"].dropna().unique()):
    try:
        info=yf.Ticker(t).get_info()
        meta.append({"Yahoo_Ticker":t,"Name":info.get("longName") or info.get("shortName"),
                     "Sector":info.get("sector"),"Industry":info.get("industry"),
                     "Quote_Type":info.get("quoteType")})
    except Exception: pass
    if (i+1)%50==0: print("metadata",i+1)
meta=pd.DataFrame(meta)
materials=meta[meta["Sector"].eq("Basic Materials")].copy()
materials["Status"]=np.where(materials["Quote_Type"].fillna("EQUITY").eq("EQUITY"),"INCLUDE","REVIEW_EXCLUDE")
materials.to_csv(DIAG/"00_materials_universe_audit.csv",index=False)
display(materials["Industry"].value_counts().head(30))
print("Provisional Materials universe:",(materials.Status=="INCLUDE").sum())

metadata 50
metadata 100
metadata 150
metadata 200
metadata 250
metadata 300
metadata 350
metadata 400
metadata 450
metadata 500
metadata 550
metadata 600


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EMUCA.AX"}}}


metadata 650
metadata 700
metadata 750
metadata 800
metadata 850
metadata 900
metadata 950
metadata 1000
metadata 1050
metadata 1100
metadata 1150
metadata 1200
metadata 1250
metadata 1300
metadata 1350
metadata 1400
metadata 1450
metadata 1500
metadata 1550
metadata 1600
metadata 1650
metadata 1700
metadata 1750
metadata 1800


Industry
Other Industrial Metals & Mining    393
Gold                                192
Other Precious Metals & Mining       95
Copper                               23
Steel                                15
Specialty Chemicals                  14
Coking Coal                           8
Agricultural Inputs                   7
Aluminum                              6
Chemicals                             4
Building Materials                    4
Lumber & Wood Production              1
Silver                                1
Name: count, dtype: int64

Provisional Materials universe: 763


## 3. Download OHLCV VALUES

In [4]:
tickers=materials.loc[materials.Status=="INCLUDE","Yahoo_Ticker"].tolist()
frames=[]
for i,t in enumerate(tickers):
    try:
        x=yf.download(t,start=START,end=END,auto_adjust=False,progress=False)
        if isinstance(x.columns,pd.MultiIndex): x.columns=x.columns.get_level_values(0)
        if x.empty: continue
        x=x.reset_index(); x["Yahoo_Ticker"]=t
        if "Adj Close" in x: x=x.rename(columns={"Adj Close":"Adj_Close"})
        frames.append(x[["Date","Yahoo_Ticker","Open","High","Low","Close","Adj_Close","Volume"]])
    except Exception as e: print("price failure",t,e)
prices=pd.concat(frames,ignore_index=True)
prices.to_parquet(RAW/"materials_ohlcv.parquet",index=False)
print(prices.Date.min(),prices.Date.max(),prices.shape)

2012-01-03 00:00:00 2026-09-18 00:00:00 (1997744, 8)


## 4. Historical shares outstanding and reconstructed market capitalisation

In [5]:
shares=[]
for i,t in enumerate(prices.Yahoo_Ticker.unique()):
    try:
        s=yf.Ticker(t).get_shares_full(start=START,end=END)
        if s is not None and len(s):
            z=s.rename("Shares_Outstanding").reset_index()
            z.columns=["Date","Shares_Outstanding"]; z["Date"]=pd.to_datetime(z["Date"]).dt.tz_localize(None)
            z["Yahoo_Ticker"]=t; shares.append(z)
    except Exception: pass
shares=pd.concat(shares,ignore_index=True) if shares else pd.DataFrame(columns=["Date","Shares_Outstanding","Yahoo_Ticker"])
shares.to_csv(RAW/"materials_historical_shares.csv",index=False)

parts=[]
for t,g in prices.groupby("Yahoo_Ticker"):
    g=g.sort_values("Date").copy()
    s=shares[shares.Yahoo_Ticker==t][["Date","Shares_Outstanding"]].sort_values("Date")
    if len(s): g=pd.merge_asof(g,s,on="Date",direction="backward")
    else: g["Shares_Outstanding"]=np.nan
    g["Market_Cap_Reconstructed"]=g["Close"]*g["Shares_Outstanding"]
    parts.append(g)
panel=pd.concat(parts,ignore_index=True)
panel.to_parquet(INTERIM/"materials_equity_panel.parquet",index=False)
print("Market-cap coverage:",panel.Market_Cap_Reconstructed.notna().mean())

Market-cap coverage: 0.6801532128240656


## 5. Acquisition diagnostics

In [6]:
a=(panel.assign(ValidPrice=lambda x:x.Adj_Close.gt(0)&x.Adj_Close.notna())
   .groupby("Yahoo_Ticker").agg(First_Date=("Date","min"),Last_Date=("Date","max"),
    Valid_Prices=("ValidPrice","sum"),MCap_Rows=("Market_Cap_Reconstructed","count"),
    Latest_MCap=("Market_Cap_Reconstructed","last")).reset_index())
a.to_csv(DIAG/"00_materials_ticker_coverage.csv",index=False)
display(a.sort_values("Latest_MCap",ascending=False).head(30))

,Yahoo_Ticker,First_Date,Last_Date,Valid_Prices,MCap_Rows,Latest_MCap
117,BHP.AX,2012-01-03,2026-09-18,3726,2784,3.103143e+11
605,RIO.AX,2012-01-03,2026-09-18,3726,2817,2.722070e+11
493,NEM.AX,2023-10-27,2026-09-18,733,693,1.853129e+11
282,FMG.AX,2012-01-03,2026-09-18,3726,2552,5.151108e+10
510,NST.AX,2012-01-03,2026-09-18,3726,2752,3.164492e+10
266,EVN.AX,2012-01-03,2026-09-18,3726,2734,2.801031e+10
628,S32.AX,2015-05-18,2026-09-18,2874,2799,2.209343e+10
373,JHX.AX,2012-01-03,2026-09-18,3726,2526,2.116005e+10
11,AAI.AX,2024-07-24,2026-09-18,548,548,1.741802e+10
430,LYC.AX,2012-01-03,2026-09-18,3726,2786,1.444331e+10
